In [1]:
import pandas as pd
import numpy as np
import os
from os.path import curdir

data_path = os.path.join(curdir,'project_data.xlsx')

client_portfolio = pd.read_excel(data_path, sheet_name="CLIENT PORTFOLIO")
deposit_account = pd.read_excel(data_path, sheet_name="DEPOSIT ACCOUNT")

print("CLIENT PORTFOLIO shape:", client_portfolio.shape)
print("DEPOSIT ACCOUNT shape:", deposit_account.shape)
print("\nCLIENT PORTFOLIO columns:", client_portfolio.columns.tolist())
print("\nDEPOSIT ACCOUNT columns:", deposit_account.columns.tolist())

CLIENT PORTFOLIO shape: (1035, 17)
DEPOSIT ACCOUNT shape: (1142, 11)

CLIENT PORTFOLIO columns: ['CLIENT_NUMBER', 'ACCOUNT_OPEN_DATE', 'OFFICE_PROV', 'OFFICE_COUNTRY', 'INDUSTRY', 'KYC_RATING', 'CREDIT_RISK_RATING', 'DEPOSIT_CURRENT_BALANCE', 'DEP_AVE_YEAR_BAL', 'DEP_INT_INCOME', 'DEP_INT_EXP', 'DEP_NET_INT_INC', 'LOAN_CUR_BAL', 'LOAN_INT_INCOME', 'LOAN_EXPENSE_INTEREST', 'LOAN_NET_INT_INC', 'FEE_INC']

DEPOSIT ACCOUNT columns: ['CLIENT_NUMBER', 'DEPOSIT_ACCOUNT_NO', 'DEMAND_TERM', 'BAL_XCA', 'TERM_START_DATE', 'TERM_END_DATE', 'TERM_DAYS', 'CLIENT_INTEREST_RATE', 'INTEREST_RATE_EXPENSE', 'MARGIN', 'NET_INTEREST_INCOME']


In [2]:
# IDs to string (never treat identifiers as numbers)
client_portfolio["CLIENT_NUMBER"] = client_portfolio["CLIENT_NUMBER"].astype("Int64").astype("string")
deposit_account["CLIENT_NUMBER"] = deposit_account["CLIENT_NUMBER"].astype("Int64").astype("string")
deposit_account["DEPOSIT_ACCOUNT_NO"] = deposit_account["DEPOSIT_ACCOUNT_NO"].astype("Int64").astype("string")

# Dates
client_portfolio["ACCOUNT_OPEN_DATE"] = pd.to_datetime(client_portfolio["ACCOUNT_OPEN_DATE"], errors="coerce")
deposit_account["TERM_START_DATE"] = pd.to_datetime(deposit_account["TERM_START_DATE"], errors="coerce")
deposit_account["TERM_END_DATE"] = pd.to_datetime(deposit_account["TERM_END_DATE"], errors="coerce")

# Numeric conversions
client_numeric_columns = [
    "DEPOSIT_CURRENT_BALANCE", "DEP_AVE_YEAR_BAL", "DEP_INT_INCOME", "DEP_INT_EXP",
    "DEP_NET_INT_INC", "LOAN_CUR_BAL", "LOAN_INT_INCOME", "LOAN_EXPENSE_INTEREST",
    "LOAN_NET_INT_INC", "FEE_INC"
]
deposit_numeric_columns = [
    "BAL_XCA", "TERM_DAYS", "CLIENT_INTEREST_RATE", "INTEREST_RATE_EXPENSE",
    "MARGIN", "NET_INTEREST_INCOME"
]

for col in client_numeric_columns:
    client_portfolio[col] = pd.to_numeric(client_portfolio[col], errors="coerce")

for col in deposit_numeric_columns:
    deposit_account[col] = pd.to_numeric(deposit_account[col], errors="coerce")

# Duplicate row check
print("Duplicate CLIENT PORTFOLIO rows:", client_portfolio.duplicated().sum())
print("Duplicate DEPOSIT ACCOUNT rows:", deposit_account.duplicated().sum())

Duplicate CLIENT PORTFOLIO rows: 0
Duplicate DEPOSIT ACCOUNT rows: 0


In [3]:
# Validating data types after conversion
client_portfolio.dtypes

CLIENT_NUMBER                      string
ACCOUNT_OPEN_DATE          datetime64[us]
OFFICE_PROV                           str
OFFICE_COUNTRY                        str
INDUSTRY                              str
KYC_RATING                            str
CREDIT_RISK_RATING                    str
DEPOSIT_CURRENT_BALANCE           float64
DEP_AVE_YEAR_BAL                  float64
DEP_INT_INCOME                    float64
DEP_INT_EXP                       float64
DEP_NET_INT_INC                   float64
LOAN_CUR_BAL                      float64
LOAN_INT_INCOME                   float64
LOAN_EXPENSE_INTEREST             float64
LOAN_NET_INT_INC                  float64
FEE_INC                           float64
dtype: object

In [4]:
# Validating data types after conversion
deposit_account.dtypes

CLIENT_NUMBER                    string
DEPOSIT_ACCOUNT_NO               string
DEMAND_TERM                         str
BAL_XCA                         float64
TERM_START_DATE          datetime64[us]
TERM_END_DATE            datetime64[us]
TERM_DAYS                       float64
CLIENT_INTEREST_RATE            float64
INTEREST_RATE_EXPENSE           float64
MARGIN                          float64
NET_INTEREST_INCOME             float64
dtype: object

In [5]:
# Strip whitespace on client-level categoricals
client_categorical_columns = ["OFFICE_PROV", "OFFICE_COUNTRY", "INDUSTRY", "KYC_RATING", "CREDIT_RISK_RATING"]
for col in client_categorical_columns:
    client_portfolio[col] = client_portfolio[col].astype("string").str.strip()

# Clean deposit type code
deposit_account["DEMAND_TERM"] = deposit_account["DEMAND_TERM"].astype("string").str.strip().str.upper()

# Replace blank strings with true missing values
client_portfolio = client_portfolio.replace(r"^\s*$", np.nan, regex=True)
deposit_account = deposit_account.replace(r"^\s*$", np.nan, regex=True)

# Descriptive deposit type label
deposit_account["DEPOSIT_TYPE"] = deposit_account["DEMAND_TERM"].map({
    "D": "Demand Deposit",
    "T": "Term Deposit"
})

print(deposit_account["DEPOSIT_TYPE"].value_counts(dropna=False))

DEPOSIT_TYPE
Demand Deposit    985
Term Deposit      157
Name: count, dtype: int64


In [6]:
print("Missing values, CLIENT PORTFOLIO:")
print(client_portfolio.isna().sum().sort_values(ascending=False))

print("\nMissing values, DEPOSIT ACCOUNT:")
print(deposit_account.isna().sum().sort_values(ascending=False))

# KYC rating: missing = not available
client_portfolio["KYC_RATING"] = client_portfolio["KYC_RATING"].fillna("Not Available")

# Credit risk rating: missing + no loan balance = rating doesn't apply (no borrowing relationship)
client_portfolio["CREDIT_RISK_RATING"] = np.where(
    client_portfolio["CREDIT_RISK_RATING"].isna() & (client_portfolio["LOAN_CUR_BAL"] == 0),
    "Not Applicable",
    client_portfolio["CREDIT_RISK_RATING"]
)
# Remaining missing ratings for clients who DO have a loan = genuinely not available
client_portfolio["CREDIT_RISK_RATING"] = client_portfolio["CREDIT_RISK_RATING"].fillna("Not Available")

# Term fields (TERM_START_DATE, TERM_END_DATE, TERM_DAYS) should only be missing for Demand deposits.
term_missing_check = deposit_account.groupby("DEPOSIT_TYPE", observed=True).agg(
    ACCOUNT_COUNT=("DEPOSIT_ACCOUNT_NO", "count"),
    MISSING_TERM_DAYS=("TERM_DAYS", lambda x: x.isna().sum())
)
print("\nTerm field missingness by deposit type:")
print(term_missing_check)

Missing values, CLIENT PORTFOLIO:
LOAN_CUR_BAL               898
LOAN_NET_INT_INC           761
LOAN_EXPENSE_INTEREST      761
LOAN_INT_INCOME            760
CREDIT_RISK_RATING         710
DEPOSIT_CURRENT_BALANCE    379
DEP_NET_INT_INC            299
DEP_INT_INCOME             299
DEP_INT_EXP                299
KYC_RATING                  44
CLIENT_NUMBER                0
DEP_AVE_YEAR_BAL             0
ACCOUNT_OPEN_DATE            0
INDUSTRY                     0
OFFICE_COUNTRY               0
OFFICE_PROV                  0
FEE_INC                      0
dtype: int64

Missing values, DEPOSIT ACCOUNT:
TERM_START_DATE          985
TERM_END_DATE            985
TERM_DAYS                985
CLIENT_NUMBER              0
DEPOSIT_ACCOUNT_NO         0
DEMAND_TERM                0
BAL_XCA                    0
CLIENT_INTEREST_RATE       0
INTEREST_RATE_EXPENSE      0
MARGIN                     0
NET_INTEREST_INCOME        0
DEPOSIT_TYPE               0
dtype: int64

Term field missingness by depo

In [7]:
# Negative value scan (informational — we keep these, negative profitability is a real finding)
for col in client_numeric_columns:
    n_neg = (client_portfolio[col] < 0).sum()
    if n_neg > 0:
        print(f"{col}: {n_neg} negative values")

# Reconciliation: does reported NII match income - expense?
client_portfolio["CALCULATED_DEP_NII"] = client_portfolio["DEP_INT_INCOME"] - client_portfolio["DEP_INT_EXP"]
client_portfolio["CALCULATED_LOAN_NII"] = client_portfolio["LOAN_INT_INCOME"] - client_portfolio["LOAN_EXPENSE_INTEREST"]
client_portfolio["DEP_NII_DIFFERENCE"] = client_portfolio["DEP_NET_INT_INC"] - client_portfolio["CALCULATED_DEP_NII"]
client_portfolio["LOAN_NII_DIFFERENCE"] = client_portfolio["LOAN_NET_INT_INC"] - client_portfolio["CALCULATED_LOAN_NII"]

print("\nMax abs DEP_NII_DIFFERENCE:", client_portfolio["DEP_NII_DIFFERENCE"].abs().max())
print("Max abs LOAN_NII_DIFFERENCE:", client_portfolio["LOAN_NII_DIFFERENCE"].abs().max())

DEP_INT_EXP: 4 negative values
DEP_NET_INT_INC: 16 negative values
LOAN_NET_INT_INC: 18 negative values
FEE_INC: 7 negative values

Max abs DEP_NII_DIFFERENCE: 0.1400000000012369
Max abs LOAN_NII_DIFFERENCE: 0.1700010001368355


In [8]:
# Checking outliers using IQR method for DEP_NET_INT_INC, LOAN_NET_INT_INC, and FEE_INC
def create_iqr_flag(data, column):
    q1, q3 = data[column].quantile(0.25), data[column].quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((data[column] < lower) | (data[column] > upper)).astype(int)

for col in ["DEP_NET_INT_INC", "LOAN_NET_INT_INC", "FEE_INC"]:
    client_portfolio[f"{col}_OUTLIER_FLAG"] = create_iqr_flag(client_portfolio, col)

# We do not automatically remove these records because major corporate clients may legitimately generate unusually large balances or revenue
print(client_portfolio[[c for c in client_portfolio.columns if "OUTLIER_FLAG" in c]].sum())

DEP_NET_INT_INC_OUTLIER_FLAG     137
LOAN_NET_INT_INC_OUTLIER_FLAG     44
FEE_INC_OUTLIER_FLAG             211
dtype: int64


In [9]:
# Breaking term days into buckets for analysis
print(deposit_account.loc[deposit_account["DEPOSIT_TYPE"] == "Term Deposit", "TERM_DAYS"].describe())

def term_bucket(days):
    if pd.isna(days):
        return "Demand"          # no term = demand deposit
    elif days <= 180:
        return "Short"            # up to 180 days
    elif days <= 365:
        return "Medium"           # up to 1 year
    else:
        return "Long"             # Greater than 1 year

deposit_account["TERM_BUCKET"] = deposit_account["TERM_DAYS"].apply(term_bucket)

print(deposit_account["TERM_BUCKET"].value_counts(dropna=False))

count     157.000000
mean      249.407643
std       177.156853
min         7.000000
25%        91.000000
50%       360.000000
75%       365.000000
max      1095.000000
Name: TERM_DAYS, dtype: float64
TERM_BUCKET
Demand    985
Medium     82
Short      56
Long       19
Name: count, dtype: int64


In [10]:
# ============================================================================
# CHUNK 8 — Aggregate deposit accounts to client level (balance-weighted)
# ============================================================================
#
# WHY THIS CHUNK IS STRUCTURED THIS WAY:
# One client can have multiple deposit accounts (e.g., 3 demand accounts +
# 2 term GICs). We need to collapse deposit_account (account-level, many
# rows per client) down to one row per client so it can merge into
# client_portfolio. But we need TWO versions of that collapse:
#   (a) an "overall" total across ALL accounts, for total balance/NII KPIs
#   (b) a "per-type" split (Demand vs Term), so the dashboard can show
#       margin/rate differences between product types instead of a single
#       blended number that hides the difference.
#
# WHY WEIGHTED AVERAGES INSTEAD OF .mean():
# CLIENT_INTEREST_RATE and MARGIN are ratios, not additive quantities.
# If a client has a $50K account at 4% and a $5M account at 1%, a simple
# .mean() says "2.5%" — which overstates what the client actually costs/
# earns the bank, because it ignores that almost all their money sits in
# the low-rate account. The correct average weights each account's rate by
# its dollar balance, so bigger accounts influence the average more.
# General formula: weighted_avg = sum(balance * rate) / sum(balance)
# ============================================================================

# --- Step 1: build a helper column so we can weight rate by balance ---
# We can't compute sum(bal*rate)/sum(bal) directly in one groupby.agg() call
# because pandas aggregates each column independently — it has no way to
# know "multiply these two columns together, THEN sum". So we pre-compute
# the product (bal * rate) as its own column here, sum THAT column in the
# groupby below, and divide by summed balance afterward. This is the
# standard trick for weighted averages in a groupby.
deposit_account["BAL_X_RATE"] = deposit_account["BAL_XCA"] * deposit_account["CLIENT_INTEREST_RATE"]


# ----------------------------------------------------------------------------
# Step 2: OVERALL client totals — collapses ALL accounts (demand + term
# combined) into one row per client. This feeds the top-level KPIs
# (Total Deposits, Total NII, blended Average Margin) where we don't care
# about product-type split, just the client's whole deposit relationship.
# ----------------------------------------------------------------------------
client_totals = deposit_account.groupby("CLIENT_NUMBER", as_index=False).agg(
    # nunique (not count) in case of any duplicate account-number rows —
    # this guarantees we're counting distinct accounts, not rows.
    DEPOSIT_ACCOUNT_COUNT=("DEPOSIT_ACCOUNT_NO", "nunique"),

    # Balances and NII are additive dollar amounts — straight sum is correct
    # here (no weighting needed, since we're not turning them into a ratio).
    TOTAL_ACCOUNT_BALANCE=("BAL_XCA", "sum"),
    TOTAL_ACCOUNT_NII=("NET_INTEREST_INCOME", "sum"),

    # Carrying the sum of (bal*rate) forward so we can finish the weighted-
    # average calculation in the next step, once we also have total balance.
    SUM_BAL_X_RATE=("BAL_X_RATE", "sum"),
)

# Now finish the weighted average: divide summed (bal*rate) by summed balance.
# This must happen AFTER the groupby, using the already-summed columns —
# doing it before would just reproduce the account-level ratio, not the
# client-level weighted one.
client_totals["AVERAGE_CLIENT_RATE"] = client_totals["SUM_BAL_X_RATE"] / client_totals["TOTAL_ACCOUNT_BALANCE"]

# Margin, similarly, is NOT averaged — it's DERIVED as total NII / total
# balance. This is mathematically identical to the weighted average of each
# account's margin, but simpler to compute directly from sums we already have.
client_totals["AVERAGE_MARGIN"] = client_totals["TOTAL_ACCOUNT_NII"] / client_totals["TOTAL_ACCOUNT_BALANCE"]

# Drop the helper column now that it's done its job — keeping it around
# would confuse anyone reading the final table (it's not a real metric).
client_totals = client_totals.drop(columns="SUM_BAL_X_RATE")

# WHY replace inf with NaN: if a client somehow has TOTAL_ACCOUNT_BALANCE
# of exactly 0 (possible if all their deposit balances are 0), dividing by
# zero produces +/-inf, not an error. Inf values would silently break any
# downstream chart or average in Looker Studio, so we convert them to NaN —
# an explicit "not calculable" rather than a nonsense number.
client_totals = client_totals.replace([np.inf, -np.inf], np.nan)


# ----------------------------------------------------------------------------
# Step 3: PER-TYPE (Demand vs Term) breakdown — this is the piece that lets
# the dashboard show "margin by product type" instead of one blended number.
# Grouping by [CLIENT_NUMBER, DEPOSIT_TYPE] together means a client with both
# demand and term accounts gets TWO rows here (one per type), each correctly
# weighted only within that type's own accounts — a term-deposit's rate never
# gets blended with a demand account's rate.
# ----------------------------------------------------------------------------
type_agg = deposit_account.groupby(["CLIENT_NUMBER", "DEPOSIT_TYPE"], as_index=False).agg(
    BALANCE=("BAL_XCA", "sum"),
    NII=("NET_INTEREST_INCOME", "sum"),
    ACCOUNT_COUNT=("DEPOSIT_ACCOUNT_NO", "nunique"),
    SUM_BAL_X_RATE=("BAL_X_RATE", "sum"),
)

# Same weighted-average logic as Step 2, just computed within each
# (client, type) group instead of within each client overall.
type_agg["WEIGHTED_RATE"] = type_agg["SUM_BAL_X_RATE"] / type_agg["BALANCE"]
type_agg["WEIGHTED_MARGIN"] = type_agg["NII"] / type_agg["BALANCE"]
type_agg = type_agg.drop(columns="SUM_BAL_X_RATE").replace([np.inf, -np.inf], np.nan)


# ----------------------------------------------------------------------------
# Step 4: PIVOT from long to wide — type_agg currently has up to 2 rows per
# client (one for Demand, one for Term). But client_portfolio has exactly
# ONE row per client, and we need to merge into it. Pivoting turns the
# DEPOSIT_TYPE row-label into column suffixes instead, so each client goes
# back to a single row, with separate columns for their demand vs term
# numbers side by side (e.g., DEMAND_BALANCE and TERM_BALANCE as two
# columns on the same row, instead of two separate rows).
# ----------------------------------------------------------------------------
type_wide = type_agg.pivot(
    index="CLIENT_NUMBER",
    columns="DEPOSIT_TYPE",
    values=["BALANCE", "NII", "ACCOUNT_COUNT", "WEIGHTED_RATE", "WEIGHTED_MARGIN"]
)

# pivot() produces messy two-level column names like ("BALANCE", "Demand Deposit").
# This renames them to flat, dashboard-friendly names like "DEMAND_BALANCE" —
# Looker Studio (and pandas generally) works much better with flat column names.
type_label = {"Demand Deposit": "DEMAND", "Term Deposit": "TERM"}
type_wide.columns = [f"{type_label[t]}_{v}" for v, t in type_wide.columns]
type_wide = type_wide.reset_index()

# WHY fillna(0) only for balance/NII/count, and NOT for rate/margin:
# If a client has ZERO term deposit accounts, the pivot leaves TERM_BALANCE
# as NaN for that client. But "no term accounts" really means the term
# balance IS zero — so filling with 0 is correct and necessary (otherwise
# summing DEMAND_BALANCE + TERM_BALANCE later would silently produce NaN
# for any client missing one type).
# We do NOT fillna(0) for TERM_WEIGHTED_RATE / TERM_WEIGHTED_MARGIN, though —
# a rate/margin of exactly 0% would be a misleading LIE for a client who
# simply has no term accounts at all (0% implies "they have term deposits
# priced at zero", not "not applicable"). Leaving these as NaN correctly
# tells the dashboard "this metric doesn't apply to this client."
for col in ["DEMAND_BALANCE", "TERM_BALANCE", "DEMAND_NII", "TERM_NII",
            "DEMAND_ACCOUNT_COUNT", "TERM_ACCOUNT_COUNT"]:
    if col in type_wide.columns:
        type_wide[col] = type_wide[col].fillna(0)


# ----------------------------------------------------------------------------
# Step 5: combine the overall totals (Step 2) with the per-type split
# (Step 4) into one client-level table. how="left" on client_totals means
# every client with at least one deposit account is guaranteed a row here,
# even in the (unlikely) case their type_wide split failed to produce one.
# ----------------------------------------------------------------------------
deposit_client_summary = client_totals.merge(type_wide, on="CLIENT_NUMBER", how="left")

print(deposit_client_summary.head())
print("\nShape:", deposit_client_summary.shape)

  CLIENT_NUMBER  DEPOSIT_ACCOUNT_COUNT  TOTAL_ACCOUNT_BALANCE  \
0   10002000011                      2           1.515688e+03   
1   10002000031                      4           5.375191e+06   
2   10002000058                      2           2.904088e+05   
3   10002000489                      1           5.490800e+08   
4   10002000553                      3           1.438012e+07   

   TOTAL_ACCOUNT_NII  AVERAGE_CLIENT_RATE  AVERAGE_MARGIN  DEMAND_BALANCE  \
0          69.025827             0.000000        0.045541    1.515688e+03   
1       34849.394069             1.767853        0.006483    5.375191e+06   
2       10896.509855             0.000000        0.037521    2.904088e+05   
3       11518.958547             3.780000        0.000021    0.000000e+00   
4      278416.721042             0.081180        0.019361    1.438012e+07   

   TERM_BALANCE     DEMAND_NII      TERM_NII  DEMAND_ACCOUNT_COUNT  \
0           0.0      69.025827      0.000000                   2.0   
1     

In [11]:
# Helper column for weighted-average math: sum(bal*rate) / sum(bal). We need to use weighted average because clients can have multiple accounts with different rates, and we want to reflect the overall effective rate for the client.
deposit_account["BAL_X_RATE"] = deposit_account["BAL_XCA"] * deposit_account["CLIENT_INTEREST_RATE"]

# Overall client totals (all accounts combined, regardless of type)
client_totals = deposit_account.groupby("CLIENT_NUMBER", as_index=False).agg(
    DEPOSIT_ACCOUNT_COUNT=("DEPOSIT_ACCOUNT_NO", "nunique"),
    TOTAL_ACCOUNT_BALANCE=("BAL_XCA", "sum"),
    TOTAL_ACCOUNT_NII=("NET_INTEREST_INCOME", "sum"),
    SUM_BAL_X_RATE=("BAL_X_RATE", "sum"),
)
client_totals["AVERAGE_CLIENT_RATE"] = client_totals["SUM_BAL_X_RATE"] / client_totals["TOTAL_ACCOUNT_BALANCE"]
client_totals["AVERAGE_MARGIN"] = client_totals["TOTAL_ACCOUNT_NII"] / client_totals["TOTAL_ACCOUNT_BALANCE"]
client_totals = client_totals.drop(columns="SUM_BAL_X_RATE").replace([np.inf, -np.inf], np.nan)

# Per-type (Demand vs Term) breakdown, still weighted
type_agg = deposit_account.groupby(["CLIENT_NUMBER", "DEPOSIT_TYPE"], as_index=False).agg(
    BALANCE=("BAL_XCA", "sum"),
    NII=("NET_INTEREST_INCOME", "sum"),
    ACCOUNT_COUNT=("DEPOSIT_ACCOUNT_NO", "nunique"),
    SUM_BAL_X_RATE=("BAL_X_RATE", "sum"),
)
type_agg["WEIGHTED_RATE"] = type_agg["SUM_BAL_X_RATE"] / type_agg["BALANCE"]
type_agg["WEIGHTED_MARGIN"] = type_agg["NII"] / type_agg["BALANCE"]
type_agg = type_agg.drop(columns="SUM_BAL_X_RATE").replace([np.inf, -np.inf], np.nan)

# Pivot to one row per client (wide), so it can merge into client_portfolio cleanly
type_wide = type_agg.pivot(index="CLIENT_NUMBER", columns="DEPOSIT_TYPE",
                            values=["BALANCE", "NII", "ACCOUNT_COUNT", "WEIGHTED_RATE", "WEIGHTED_MARGIN"])

type_label = {"Demand Deposit": "DEMAND", "Term Deposit": "TERM"}
type_wide.columns = [f"{type_label[t]}_{v}" for v, t in type_wide.columns]
type_wide = type_wide.reset_index()

# Balances/NII/counts: no accounts of that type = 0, not missing. Rates/margins stay NaN if not applicable.
for col in ["DEMAND_BALANCE", "TERM_BALANCE", "DEMAND_NII", "TERM_NII", "DEMAND_ACCOUNT_COUNT", "TERM_ACCOUNT_COUNT"]:
    if col in type_wide.columns:
        type_wide[col] = type_wide[col].fillna(0)

# Combine overall totals + per-type split into one client-level summary
deposit_client_summary = client_totals.merge(type_wide, on="CLIENT_NUMBER", how="left")

print(deposit_client_summary.head())
print("\nShape:", deposit_client_summary.shape)

  CLIENT_NUMBER  DEPOSIT_ACCOUNT_COUNT  TOTAL_ACCOUNT_BALANCE  \
0   10002000011                      2           1.515688e+03   
1   10002000031                      4           5.375191e+06   
2   10002000058                      2           2.904088e+05   
3   10002000489                      1           5.490800e+08   
4   10002000553                      3           1.438012e+07   

   TOTAL_ACCOUNT_NII  AVERAGE_CLIENT_RATE  AVERAGE_MARGIN  DEMAND_BALANCE  \
0          69.025827             0.000000        0.045541    1.515688e+03   
1       34849.394069             1.767853        0.006483    5.375191e+06   
2       10896.509855             0.000000        0.037521    2.904088e+05   
3       11518.958547             3.780000        0.000021    0.000000e+00   
4      278416.721042             0.081180        0.019361    1.438012e+07   

   TERM_BALANCE     DEMAND_NII      TERM_NII  DEMAND_ACCOUNT_COUNT  \
0           0.0      69.025827      0.000000                   2.0   
1     

In [12]:
# Summary sanity check
deposit_segment_summary = deposit_account.groupby("TERM_BUCKET", as_index=False).agg(
    ACCOUNT_COUNT=("DEPOSIT_ACCOUNT_NO", "nunique"),
    TOTAL_BALANCE=("BAL_XCA", "sum"),
    TOTAL_NII=("NET_INTEREST_INCOME", "sum"),
    SUM_BAL_X_RATE=("BAL_X_RATE", "sum"),
)
deposit_segment_summary["WEIGHTED_AVG_RATE"] = deposit_segment_summary["SUM_BAL_X_RATE"] / deposit_segment_summary["TOTAL_BALANCE"]
deposit_segment_summary["WEIGHTED_AVG_MARGIN"] = deposit_segment_summary["TOTAL_NII"] / deposit_segment_summary["TOTAL_BALANCE"]
deposit_segment_summary = deposit_segment_summary.drop(columns="SUM_BAL_X_RATE")

print(deposit_segment_summary)

  TERM_BUCKET  ACCOUNT_COUNT  TOTAL_BALANCE     TOTAL_NII  WEIGHTED_AVG_RATE  \
0      Demand            985   3.348679e+08  5.118340e+06           0.675946   
1        Long             19   1.207653e+07  3.362858e+04           3.077204   
2      Medium             82   3.146655e+08  2.676084e+05           3.237007   
3       Short             56   8.798124e+08  1.245827e+05           3.858664   

   WEIGHTED_AVG_MARGIN  
0             0.015285  
1             0.002785  
2             0.000850  
3             0.000142  


It's an inverted yield curve!!! What????

In [13]:
integrated_data = pd.merge(
    client_portfolio,
    deposit_client_summary,
    on="CLIENT_NUMBER",
    how="outer",
    indicator=True
)
print(integrated_data["_merge"].value_counts())

# Fill balances with 0 for relationship classification (missing ≠ has a relationship)
balance_cols = ["DEPOSIT_CURRENT_BALANCE", "TOTAL_ACCOUNT_BALANCE", "LOAN_CUR_BAL"]
integrated_data[balance_cols] = integrated_data[balance_cols].fillna(0)

integrated_data["HAS_DEPOSIT"] = (
    (integrated_data["DEPOSIT_CURRENT_BALANCE"] > 0)
    | (integrated_data["TOTAL_ACCOUNT_BALANCE"] > 0)
    | (integrated_data["DEPOSIT_ACCOUNT_COUNT"].fillna(0) > 0)
).astype(int)

integrated_data["HAS_LOAN"] = (integrated_data["LOAN_CUR_BAL"] > 0).astype(int)

conditions = [
    (integrated_data["HAS_DEPOSIT"] == 1) & (integrated_data["HAS_LOAN"] == 1),
    (integrated_data["HAS_DEPOSIT"] == 1) & (integrated_data["HAS_LOAN"] == 0),
    (integrated_data["HAS_DEPOSIT"] == 0) & (integrated_data["HAS_LOAN"] == 1),
]
integrated_data["RELATIONSHIP_TYPE"] = np.select(
    conditions,
    ["Full Relationship", "Deposit Only", "Loan Only"],
    default="No Active Balance"
)

integrated_data["DEPOSIT_ONLY_FLAG"] = (integrated_data["RELATIONSHIP_TYPE"] == "Deposit Only").astype(int)
integrated_data["LOAN_ONLY_FLAG"] = (integrated_data["RELATIONSHIP_TYPE"] == "Loan Only").astype(int)
integrated_data["FULL_RELATIONSHIP_FLAG"] = (integrated_data["RELATIONSHIP_TYPE"] == "Full Relationship").astype(int)

print(integrated_data["RELATIONSHIP_TYPE"].value_counts())
print("Final shape:", integrated_data.shape)

_merge
both          643
left_only     392
right_only      1
Name: count, dtype: int64
RELATIONSHIP_TYPE
Deposit Only         627
No Active Balance    334
Loan Only             58
Full Relationship     17
Name: count, dtype: int64
Final shape: (1036, 46)


In [14]:
# Feature-Engineering here on

# --- Profitability (revenue-based: NII + Fees; no cost-to-serve data available) ---
integrated_data["TOTAL_NII"] = integrated_data["DEP_NET_INT_INC"].fillna(0) + integrated_data["LOAN_NET_INT_INC"].fillna(0)
integrated_data["TOTAL_PROFITABILITY"] = integrated_data["TOTAL_NII"] + integrated_data["FEE_INC"].fillna(0)

# --- Credit risk ordinal score (AAA=safest -> D=riskiest; confirm scale matches your bank) ---
credit_risk_scale = {
    "AAA": 1, "AA": 2, "A": 3, "BBB+": 4, "BBB": 5, "BBB-": 6,
    "BB+": 7, "BB": 8, "BB-": 9, "B+": 10, "B": 11, "B-": 12,
    "CCC": 13, "CC": 14, "D": 15
    # "Not Applicable" / "Not Available" intentionally left unmapped -> NaN
}
integrated_data["CREDIT_RISK_SCORE"] = integrated_data["CREDIT_RISK_RATING"].map(credit_risk_scale)

# --- KYC ordinal score (A=lowest AML risk -> E=highest; confirm scale matches your bank) ---
kyc_scale = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5}
integrated_data["KYC_RISK_SCORE"] = integrated_data["KYC_RATING"].map(kyc_scale)

# --- Client tenure (assumes "today" as snapshot date — replace with your actual as-of date if you have one) ---
snapshot_date = pd.Timestamp.today()
integrated_data["CLIENT_TENURE_YEARS"] = (snapshot_date - integrated_data["ACCOUNT_OPEN_DATE"]).dt.days / 365.25

# --- Balance tier (relationship size, for concentration/pricing views) ---
integrated_data["CLIENT_TOTAL_BALANCE"] = integrated_data["DEPOSIT_CURRENT_BALANCE"].fillna(0) + integrated_data["LOAN_CUR_BAL"].fillna(0)

def balance_tier(bal):
    if bal < 100_000:
        return "< $100K"
    elif bal < 1_000_000:
        return "$100K - $1M"
    else:
        return "> $1M"

integrated_data["BALANCE_TIER"] = integrated_data["CLIENT_TOTAL_BALANCE"].apply(balance_tier)

print(integrated_data[["TOTAL_NII", "TOTAL_PROFITABILITY", "CREDIT_RISK_SCORE", "KYC_RISK_SCORE",
                        "CLIENT_TENURE_YEARS", "BALANCE_TIER"]].describe(include="all"))

           TOTAL_NII  TOTAL_PROFITABILITY  CREDIT_RISK_SCORE  KYC_RISK_SCORE  \
count   1.036000e+03         1.036000e+03         325.000000      991.000000   
unique           NaN                  NaN                NaN             NaN   
top              NaN                  NaN                NaN             NaN   
freq             NaN                  NaN                NaN             NaN   
mean    4.874367e+04         7.956901e+04           8.049231        2.180626   
std     2.897927e+05         3.636432e+05           2.535679        1.291256   
min    -2.004808e+06        -1.877525e+06           1.000000        1.000000   
25%     6.248325e+00         1.592825e+02           6.000000        1.000000   
50%     3.012398e+02         1.082478e+03           8.000000        2.000000   
75%     5.654600e+03         1.638376e+04           9.000000        3.000000   
max     6.381429e+06         7.702800e+06          15.000000        5.000000   

        CLIENT_TENURE_YEARS BALANCE_TIE

In [16]:
print("Final integrated dataset shape:", integrated_data.shape)
print("Duplicate client IDs:", integrated_data["CLIENT_NUMBER"].duplicated().sum())
print("Missing client IDs:", integrated_data["CLIENT_NUMBER"].isna().sum())

Final integrated dataset shape: (1036, 53)
Duplicate client IDs: 0
Missing client IDs: 0


In [15]:
print(integrated_data["TOTAL_PROFITABILITY"].median())
print(integrated_data["CREDIT_RISK_SCORE"].median())

1082.4784920000002
8.0


In [17]:
output_path = os.path.join(curdir,'clean_data_w_features1.xlsx')

with pd.ExcelWriter(
    output_path,
    engine="openpyxl"
) as writer:

    client_portfolio.to_excel(
        writer,
        sheet_name="CLEAN_CLIENT_PORTFOLIO",
        index=False
    )

    deposit_account.to_excel(
        writer,
        sheet_name="CLEAN_DEPOSIT_ACCOUNT",
        index=False
    )

    deposit_client_summary.to_excel(
        writer,
        sheet_name="DEPOSIT_CLIENT_SUMMARY",
        index=False
    )

    integrated_data.to_excel(
        writer,
        sheet_name="INTEGRATED_DATA",
        index=False
    )

print("Cleaned file saved to:")
print(output_path)

Cleaned file saved to:
./clean_data_w_features1.xlsx
